In [1]:
import os

os.makedirs("src", exist_ok=True)

In [2]:
import os

os.makedirs("src", exist_ok=True)

memory_code = """
import numpy as np
from collections import deque


class MemoryBuffer:

    def __init__(self, max_size=5000):
        self.memory = deque(maxlen=max_size)

    def add_experience(
        self,
        state,
        action,
        reward,
        next_state,
        done,
        episode_reward=0
    ):

        experience = {
            "state": np.array(state),
            "action": int(action),
            "reward": float(reward),
            "next_state": np.array(next_state),
            "done": bool(done),
            "episode_reward": float(episode_reward)
        }

        self.memory.append(experience)

    def size(self):
        return len(self.memory)

    def clear(self):
        self.memory.clear()

    def get_all(self):
        return list(self.memory)

    def sample(self, batch_size=32):

        if len(self.memory) == 0:
            return []

        batch_size = min(batch_size, len(self.memory))

        idx = np.random.choice(
            len(self.memory),
            batch_size,
            replace=False
        )

        return [self.memory[i] for i in idx]

    def get_top_experiences(self, top_k=50):

        if len(self.memory) == 0:
            return []

        sorted_memory = sorted(
            self.memory,
            key=lambda x: x["episode_reward"],
            reverse=True
        )

        return sorted_memory[:top_k]

    def save(self, filepath):

        np.save(
            filepath,
            list(self.memory),
            allow_pickle=True
        )

    def load(self, filepath):

        loaded = np.load(
            filepath,
            allow_pickle=True
        )

        self.memory = deque(
            loaded.tolist(),
            maxlen=self.memory.maxlen
        )
"""

with open("src/memory.py", "w") as f:
    f.write(memory_code)

print("memory.py created successfully")

memory.py created successfully


In [3]:
import numpy as np

from src.memory import MemoryBuffer

In [4]:
memory = MemoryBuffer(
    max_size=1000
)

In [5]:
for i in range(100):

    state = np.random.rand(20)

    action = np.random.randint(0, 7)

    reward = np.random.rand()

    next_state = np.random.rand(20)

    done = False

    episode_reward = np.random.rand() * 10

    memory.add_experience(
        state,
        action,
        reward,
        next_state,
        done,
        episode_reward
    )

In [6]:
print(
    "Memory Size:",
    memory.size()
)

Memory Size: 100


In [7]:
top = memory.get_top_experiences(
    top_k=5
)

for item in top:

    print(
        item["episode_reward"]
    )

9.934604282455796
9.819095657467871
9.815381508573894
9.783682042023765
9.741777289402906


In [8]:
import importlib
import src.memory

importlib.reload(src.memory)

from src.memory import MemoryBuffer

In [15]:
import os

file_path = "/results"

# Create parent directories if they don't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

In [16]:
memory.save(
    "src/memory.npy"
)

print("Memory Saved")

Memory Saved


In [17]:
import os

os.makedirs("src", exist_ok=True)

retrieval_code = """
import numpy as np


class MemoryRetriever:

    def __init__(self, memory_buffer):

        self.memory_buffer = memory_buffer

    def cosine_similarity(
        self,
        state1,
        state2
    ):

        state1 = np.array(state1).flatten()
        state2 = np.array(state2).flatten()

        denominator = (
            np.linalg.norm(state1)
            * np.linalg.norm(state2)
        )

        if denominator == 0:
            return 0

        return np.dot(
            state1,
            state2
        ) / denominator

    def retrieve_best_match(
        self,
        current_state
    ):

        memories = self.memory_buffer.get_all()

        if len(memories) == 0:
            return None

        best_memory = None
        best_score = -1

        for memory in memories:

            similarity = self.cosine_similarity(
                current_state,
                memory["state"]
            )

            if similarity > best_score:

                best_score = similarity
                best_memory = memory

        return best_memory

    def retrieve_top_k(
        self,
        current_state,
        k=5
    ):

        memories = self.memory_buffer.get_all()

        if len(memories) == 0:
            return []

        scores = []

        for memory in memories:

            similarity = self.cosine_similarity(
                current_state,
                memory["state"]
            )

            scores.append(
                (
                    similarity,
                    memory
                )
            )

        scores.sort(
            reverse=True,
            key=lambda x: x[0]
        )

        return scores[:k]
"""

with open("src/retrieval.py", "w") as f:
    f.write(retrieval_code)

print("retrieval.py created")

retrieval.py created


In [18]:
import importlib

import src.retrieval

importlib.reload(src.retrieval)

from src.retrieval import MemoryRetriever

In [19]:
retriever = MemoryRetriever(
    memory
)

print("Retriever Ready")

Retriever Ready


In [20]:
query_state = np.random.rand(20)

In [21]:
best = retriever.retrieve_best_match(
    query_state
)

print(best.keys())

dict_keys(['state', 'action', 'reward', 'next_state', 'done', 'episode_reward'])


In [22]:
top5 = retriever.retrieve_top_k(
    query_state,
    k=5
)

print(
    "Retrieved:",
    len(top5)
)

Retrieved: 5


In [23]:
import os

os.makedirs("src", exist_ok=True)

intrinsic_reward_code = """
import numpy as np


class IntrinsicReward:

    def __init__(
        self,
        threshold=0.95,
        bonus_reward=0.1
    ):

        self.threshold = threshold
        self.bonus_reward = bonus_reward

        self.visited_states = []

    def cosine_similarity(
        self,
        state1,
        state2
    ):

        state1 = np.array(state1).flatten()
        state2 = np.array(state2).flatten()

        norm1 = np.linalg.norm(state1)
        norm2 = np.linalg.norm(state2)

        if norm1 == 0 or norm2 == 0:
            return 0

        return np.dot(
            state1,
            state2
        ) / (norm1 * norm2)

    def is_novel(
        self,
        state
    ):

        if len(self.visited_states) == 0:
            return True

        similarities = []

        for visited in self.visited_states:

            similarity = self.cosine_similarity(
                state,
                visited
            )

            similarities.append(similarity)

        max_similarity = max(similarities)

        return max_similarity < self.threshold

    def compute_bonus(
        self,
        state
    ):

        if self.is_novel(state):

            self.visited_states.append(
                np.array(state)
            )

            return self.bonus_reward

        return 0.0

    def reset(self):

        self.visited_states = []
"""

with open("src/intrinsic_reward.py", "w") as f:
    f.write(intrinsic_reward_code)

print("intrinsic_reward.py created")

intrinsic_reward.py created


In [24]:
import importlib

import src.intrinsic_reward

importlib.reload(src.intrinsic_reward)

from src.intrinsic_reward import IntrinsicReward

print("IntrinsicReward imported")

IntrinsicReward imported


In [25]:
novelty = IntrinsicReward(
    threshold=0.95,
    bonus_reward=0.1
)

In [26]:
import numpy as np

state1 = np.random.rand(20)

bonus = novelty.compute_bonus(
    state1
)

print("Bonus:", bonus)

Bonus: 0.1


In [27]:
bonus = novelty.compute_bonus(
    state1
)

print("Bonus:", bonus)

Bonus: 0.0


In [28]:
state2 = np.random.rand(20)

bonus = novelty.compute_bonus(
    state2
)

print("Bonus:", bonus)

Bonus: 0.1
